Task:
1. Load with pandas; check class balance of left_in_next_60d.

2. Modeling:

Use numeric features (tenure_months, monthly_spend, logins_last_30d, tickets_last_90d) plus categorical (plan_type, country via one‑hot).

Train a simple model (LogisticRegression or RandomForestClassifier) with train/test split.

Get accuracy and confusion matrix.

3. Feature importance / reasoning:

If using RandomForest: inspect feature importances.

If using LogisticRegression: inspect coefficients (sign + magnitude).

Optionally, cluster feature importances of login vs ticket vs spend vs tenure.

4. GenAI element (conceptual, not heavy):

Use a Hugging Face text‑generation or explanation model (e.g., a text-generation or text2text-generation pipeline) to generate a natural‑language description of the “likely churner profile” given the most important features (you provide a short prompt summarizing the numeric findings).

7. In markdown/text, write 3–4 sentences summarizing:

Which behavioural signals (low logins, high tickets, short tenure, high spend, etc.) align with higher churn in this sample.

How you would turn this into rules or alerts for a success team (e.g., “customers with X and Y should get a proactive call”).

In [1]:
import pandas as pd
import numpy as np

from io import StringIO

data = """customer_id,tenure_months,monthly_spend,logins_last_30d,tickets_last_90d,plan_type,country,left_in_next_60d
1,3,1200,4,2,Basic,IN,1
2,18,800,15,0,Standard,US,0
3,6,950,10,1,Basic,IN,1
4,24,700,20,0,Premium,UK,0
5,12,1100,8,3,Standard,IN,1
6,30,650,25,0,Premium,US,0
7,2,1300,3,4,Basic,IN,1
8,20,900,18,1,Standard,UK,0
9,10,1000,6,2,Standard,US,1
10,36,600,28,0,Premium,IN,0"""

df = pd.read_csv(StringIO(data))

df

,customer_id,tenure_months,monthly_spend,logins_last_30d,tickets_last_90d,plan_type,country,left_in_next_60d
0,1,3,1200,4,2,Basic,IN,1
1,2,18,800,15,0,Standard,US,0
2,3,6,950,10,1,Basic,IN,1
3,4,24,700,20,0,Premium,UK,0
4,5,12,1100,8,3,Standard,IN,1
5,6,30,650,25,0,Premium,US,0
6,7,2,1300,3,4,Basic,IN,1
7,8,20,900,18,1,Standard,UK,0
8,9,10,1000,6,2,Standard,US,1
9,10,36,600,28,0,Premium,IN,0


In [2]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       10 non-null     int64 
 1   tenure_months     10 non-null     int64 
 2   monthly_spend     10 non-null     int64 
 3   logins_last_30d   10 non-null     int64 
 4   tickets_last_90d  10 non-null     int64 
 5   plan_type         10 non-null     object
 6   country           10 non-null     object
 7   left_in_next_60d  10 non-null     int64 
dtypes: int64(6), object(2)
memory usage: 772.0+ bytes


In [4]:
df.describe()

,customer_id,tenure_months,monthly_spend,logins_last_30d,tickets_last_90d,left_in_next_60d
count,10.00000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,5.50000,16.100000,920.000000,13.700000,1.300000,0.500000
std,3.02765,11.532081,235.937845,8.857514,1.418136,0.527046
min,1.00000,2.000000,600.000000,3.000000,0.000000,0.000000
25%,3.25000,7.000000,725.000000,6.500000,0.000000,0.000000
50%,5.50000,15.000000,925.000000,12.500000,1.000000,0.500000
75%,7.75000,23.000000,1075.000000,19.500000,2.000000,1.000000
max,10.00000,36.000000,1300.000000,28.000000,4.000000,1.000000


In [5]:
df.isnull().sum()

customer_id         0
tenure_months       0
monthly_spend       0
logins_last_30d     0
tickets_last_90d    0
plan_type           0
country             0
left_in_next_60d    0
dtype: int64

In [11]:
X = df.drop(columns = 'left_in_next_60d',axis=1)
y = df['left_in_next_60d']

In [21]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numerical_columns = ["tenure_months","monthly_spend", "logins_last_30d","tickets_last_90d"]
categorical_columns = ["plan_type","country"]

preprocessing = ColumnTransformer(
    transformers = [("numerical",StandardScaler(),numerical_columns),
                    ("Categorical",OneHotEncoder(handle_unknown = 'ignore'),categorical_columns)]
)

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.3,random_state = 42)



steps = [("preprocess",preprocessing),
         ("classification", LogisticRegression())]

pipe1 = Pipeline(steps = steps, verbose = True)


In [22]:
pipe1

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numerical_categorical',
                                                  StandardScaler(),
                                                  ['tenure_months',
                                                   'monthly_spend',
                                                   'logins_last_30d',
                                                   'tickets_last_90d']),
                                                 ('Categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['plan_type', 'country'])])),
                ('classification', LogisticRegression())],
         verbose=True)

In [23]:
pipe1.fit(X_train,y_train)
y_pred = pipe1.predict(X_test)

[Pipeline] ........ (step 1 of 2) Processing preprocess, total=   0.0s
[Pipeline] .... (step 2 of 2) Processing classification, total=   0.0s


In [24]:
y_pred

array([1, 0, 0], dtype=int64)

In [25]:
y_test

8    1
1    0
5    0
Name: left_in_next_60d, dtype: int64

In [26]:
from sklearn.metrics import accuracy_score, confusion_matrix
print("Accuracy:",accuracy_score(y_test,y_pred))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))

Accuracy: 1.0
Confusion Matrix:
 [[2 0]
 [0 1]]


In [29]:
feature_names = numerical_columns + list(pipe1.named_steps['preprocess'].named_transformers_['Categorical'].get_feature_names_out(categorical_columns))

feature_names

['tenure_months',
 'monthly_spend',
 'logins_last_30d',
 'tickets_last_90d',
 'plan_type_Basic',
 'plan_type_Premium',
 'plan_type_Standard',
 'country_IN',
 'country_UK']

In [36]:
coef = pipe1.named_steps['classification'].coef_[0]
coef

array([-0.58113844,  0.44951208, -0.59053968,  0.42570029,  0.27963628,
       -0.1673541 , -0.11209517,  0.36583614, -0.36564913])

In [50]:
importance = pd.DataFrame({'feature':feature_names,'Coef':coef}).sort_values('Coef',ascending = False)
importance

,feature,Coef
1,monthly_spend,0.449512
3,tickets_last_90d,0.425700
7,country_IN,0.365836
4,plan_type_Basic,0.279636
6,plan_type_Standard,-0.112095
5,plan_type_Premium,-0.167354
8,country_UK,-0.365649
0,tenure_months,-0.581138
2,logins_last_30d,-0.590540


In [40]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-base")

Device set to use cpu


In [48]:
prompt = """Based on churn analysis, the likely churner profile is:
- High monthly spend
- Frequent support tickets
- Short tenure
- Few logins
- Basic plan users
- More common in India

Write a 1-2 sentence business‑friendly description of this customer profile, explaining why these factors increase churn risk.
"""

In [49]:
result = generator(prompt, max_length = 400)
print(result)

[{'generated_text': 'High monthly spend, frequent support tickets, and short tenure are all factors that increase churn risk.'}]


1. Which behavioural signals (low logins, high tickets, short tenure, high spend, etc.) align with higher churn in this sample.

Customers most at risk of churn in this sample show low engagement (few logins), high frustration (many support tickets), short tenure, and paradoxically higher monthly spend. These signals suggest they are paying but not finding value, often raising issues and leaving early.

2. How you would turn this into rules or alerts for a success team (e.g., “customers with X and Y should get a proactive call”).

To turn this into actionable rules for a success team, you could set alerts such as: “Flag customers with tenure under 6 months AND more than 2 tickets in the last 90 days” or “Trigger a proactive call for high‑spend customers who log in fewer than 5 times per month.” This way, the team can intervene early with targeted outreach before these customers decide to leave.